In [ ]:
%cd ..

In [ ]:
import os
import glob
import pandas as pd

from state_predictor.coder import Coder
from src.model.yolo_handler import YoloHandler
from utils.config_parser import ConfigParser
from utils.common_utils import normalize, denormalize


# Init report
report_df = pd.DataFrame(columns=["importance", "target_spars", "dataset", "original_params", "original_mAP", "original_gflops", 
                                   "pruned_params", "pruned_mAP", "pruned_mAP_drop", "pruned_spars", "pruned_dmap", "pruned_spars/dmap", "pruned_gflops", "pruned_speedup",
                                   "finetune_epochs", "finetune_params", "finetune_mAP", "finetune_mAP_drop", "finetune_spars", "finetune_dmap", "finetune_spars/dmap", "finetune_gflops"
                                   ])


## Init 

In [ ]:
dataset = "kitti"
importance = "magnitude" # [magnitude, hessian, lamp, random, bn_scale]
target_spars = 0.45
ft_epocshs = 1

model_conf = ConfigParser.read(f"config/model/{dataset}_model.ini")
yolo_handler = YoloHandler(model_conf.model)

# # Evaluate original model
init_metrics = yolo_handler.evaluate()

report_df["dataset"] = [os.path.basename(model_conf.model.data)]
print(report_df["dataset"])
report_df["importance"] = [importance]
report_df["target_spars"] = [target_spars]

report_df["original_params"] = [init_metrics[4]]
report_df["original_mAP"] = [init_metrics[2]*100]
report_df["original_gflops"] = [round(init_metrics[5], 1)]


## Prune & Evaluate

In [ ]:
# # Prune model
yolo_handler.prune_magnitude(importance=importance, target_spars=target_spars)
pruned_metrics = yolo_handler.evaluate()

# Define coder
sample_data_path_ex = "/data2/blanka/DATASETS/SPN/YOLOv8x/data"
sample_pattern_ex = "0_*.pkl"
data_files_ex = glob.glob(os.path.join(sample_data_path_ex, sample_pattern_ex))
data_file_ex = data_files_ex[0]
example_state_df = pd.read_pickle(data_file_ex)

coder = Coder(state_example=example_state_df, label_example=None, alpha_range=[0.0, 2.2])

# Construct label
metric_features = ['recall', 'precision', 'map50', 'map90', 'n_params', 'gflops']
init_columns = [col + '_init' for col in metric_features]
label_df = pd.DataFrame([],columns=metric_features + ["n_layer_channels"] + init_columns)

label_df.loc[0, metric_features] = pruned_metrics
label_df.loc[0, 'n_layer_channels'] = 0
label_df.loc[0, init_columns] = init_metrics

encoded_metrics = coder.encode_label(label_df)
decoded_true_spars = denormalize(encoded_metrics[0], value_range=(0, 1))
decoded_true_dmap = denormalize(encoded_metrics[1], value_range=(0, 1))

print(f"Layer .. spars: {decoded_true_spars:.3f}\t dmap: {decoded_true_dmap:.3f}")

# Logging for the Report
report_df["pruned_params"] = [pruned_metrics[4]]
report_df["pruned_mAP"] = [pruned_metrics[2]*100]
report_df["pruned_spars"] = [round(decoded_true_spars.item()*100, 2)]
report_df["pruned_dmap"] = [round(decoded_true_dmap.item()*100, 2)]
report_df["pruned_gflops"] = [round(pruned_metrics[5], 1)]


In [ ]:
yolo_handler.save_pruned_model(path=f"/data2/blanka/MODELS/PRUNED/pruned_spars{report_df["pruned_spars"].item()}_dmap{report_df["pruned_dmap"].item()}_{importance}.pt")

## Fine-tuning

In [ ]:
yolo_handler.fine_tune(data_yaml=f"/home/blanka/Multi-Domain-Pruning/config/data/{dataset}.yaml", epochs=ft_epocshs)

# Evaluate fine-tuned model
ft_metrics = yolo_handler.evaluate()

# Construct label
metric_features = ['recall', 'precision', 'map50', 'map90', 'n_params', 'gflops']
init_columns = [col + '_init' for col in metric_features]
ft_label_df = pd.DataFrame([],columns=metric_features + ["n_layer_channels"] + init_columns)

ft_label_df.loc[0, metric_features] = ft_metrics
ft_label_df.loc[0, 'n_layer_channels'] = 0
ft_label_df.loc[0, init_columns] = init_metrics

encoded_ft_metrics = coder.encode_label(ft_label_df)
decoded_ft_true_spars = denormalize(encoded_ft_metrics[0], value_range=(0, 1))
decoded_ft_true_dmap = denormalize(encoded_ft_metrics[1], value_range=(0, 1))

print(f"Layer .. finetune spars: {decoded_ft_true_spars:.3f}\t finetune dmap: {decoded_ft_true_dmap:.3f}")
print(ft_metrics)

# Logging for the Report
report_df["finetune_epochs"] = [ft_epocshs]
report_df["finetune_params"] = [ft_metrics[4]]
report_df["finetune_mAP"] = [ft_metrics[2]*100]
report_df["finetune_spars"] = [round(decoded_ft_true_spars.item()*100, 2)]
report_df["finetune_dmap"] = [round(decoded_ft_true_dmap.item()*100, 2)]
report_df["finetune_gflops"] = [round(ft_metrics[5], 1)]


## Save Report

In [ ]:
######### CALCULATE missing metrics for the Report  #########

def is_filled(*cols):
    return all(c in report_df and report_df[c].notna().any() for c in cols)

if is_filled("original_mAP", "pruned_mAP"):
    report_df["pruned_mAP_drop"] = report_df["original_mAP"] - report_df["pruned_mAP"]

if is_filled("original_gflops", "pruned_gflops"):
    report_df["pruned_speedup"] = round((1-report_df["pruned_gflops"] / report_df["original_gflops"])*100, 2)

if is_filled("pruned_spars", "pruned_dmap"):
    report_df["pruned_spars/dmap"] = round(report_df["pruned_spars"] / report_df["pruned_dmap"], 2)

if is_filled("original_mAP", "finetune_mAP"):
    report_df["finetune_mAP_drop"] = report_df["original_mAP"] - report_df["finetune_mAP"]

if is_filled("finetune_spars", "finetune_dmap"):
    report_df["finetune_spars/dmap"] = round(report_df["finetune_spars"] / report_df["finetune_dmap"], 2)

######### PRINT ###########  
  
pd.set_option('display.max_rows', None)      # Show all rows
pd.set_option('display.max_columns', None)   # Show all columns
pd.set_option('display.width', None)         # Don't wrap lines
pd.set_option('display.max_colwidth', None)  # Show full contents of each cell

print(report_df)


######### SAVE ###########  

base_path = "/data2/blanka/REPORTS/YOLOv8x"
dataset_part = report_df["dataset"].str.replace(r"\.yaml$", "", regex=True).iloc[0]
report_path = os.path.join(base_path, f"{importance}_{target_spars}_{dataset_part}_ft{ft_epocshs}.csv")


# Only these columns will be overwritten if file already exists
# Set [] to update all the df even if it exists
cols_to_update = ["original_gflops", "pruned_gflops", "finetune_gflops", "pruned_speedup"] #  

if os.path.exists(report_path) and len(cols_to_update):
    existing_df = pd.read_csv(report_path)

    # Ensure existing df has all columns from report_df
    for col in report_df.columns:
        if col not in existing_df.columns:
            existing_df[col] = pd.NA

    # Update only selected columns
    for col in cols_to_update:
        if col in report_df.columns:
            existing_df[col] = report_df[col]

    # Keep column order defined by report_df
    final_df = existing_df[report_df.columns]
    print(f"Columns {cols_to_update} updated in report {report_path}")

else:
    final_df = report_df.copy()  
    print(f"Report is saved to {report_path}")


final_df.to_csv(report_path, index=False)
